# Fine-tune ViT5 cho Medical Query Rewrite (ASR Error Correction)
- **Môi trường**: Kaggle (2x T4 GPU hoặc P100)
- **Model**: `VietAI/vit5-base`
- **Cấu trúc**: Phân chia theo từng cell riêng biệt (chuẩn cấu trúc từ notebook tham khảo NLP `05-train-vit5-rewrite`)

In [1]:
# Cài đặt các phiên bản ổn định (theo chuẩn file tham khảo 05-train-vit5-rewrite)
# Lưu ý: BẮT BUỘC phải cài peft==0.10.0 đi kèm accelerate<0.29.0 để tránh lỗi import clear_device_cache
!pip install -q -U "transformers<4.40.0" "tokenizers<0.19.0" "peft==0.10.0" "accelerate<0.29.0" protobuf datasets evaluate sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grp

In [2]:
# Đăng nhập Hugging Face Hub (Sử dụng secret HF_TOKEN_READ trên Kaggle)
import os
if os.path.exists("/kaggle/working"):
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    try:
        hf_token = UserSecretsClient().get_secret("HF_TOKEN_READ")
        login(token=hf_token)
        print("[INFO] Successfully authenticated with HuggingFace Hub using HF_TOKEN_READ.")
    except Exception:
        print("[WARNING] Could not load HF_TOKEN_READ from Kaggle Secrets. Proceeding anonymously.")

[INFO] Successfully authenticated with HuggingFace Hub using HF_TOKEN_READ.


In [3]:
# Import các thư viện và cấu hình tham số huấn luyện
import time
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback
)

MODEL_ID = "VietAI/vit5-base"
TRAIN_JSON_PATH = "/kaggle/input/datasets/hdtuznn/dataset-for-finetuned-vit5/finetuned_ViT5/train_vit5.json"
VAL_JSON_PATH = "/kaggle/input/datasets/hdtuznn/dataset-for-finetuned-vit5/finetuned_ViT5/val_vit5.json"
OUTPUT_DIR = "/kaggle/working/vit5_medical_rewrite_checkpoints"

MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 16
MAX_TRAIN_SAMPLES = 100000
MAX_VAL_SAMPLES = 5000

print(f"[INFO] Configuration loaded. Base model: {MODEL_ID}")

[INFO] Configuration loaded. Base model: VietAI/vit5-base


In [4]:
# Tải dữ liệu huấn luyện và kiểm thử từ file JSON
print(f"[INFO] Loading training data...")
if os.path.exists(TRAIN_JSON_PATH):
    df_train = pd.read_json(TRAIN_JSON_PATH)
    df_val = pd.read_json(VAL_JSON_PATH)
else:
    df_train = pd.read_json("/kaggle/input/datasets/hdtuznn/dataset-for-finetuned-vit5/finetuned_ViT5/train_vit5.json")
    df_val = pd.read_json("/kaggle/input/datasets/hdtuznn/dataset-for-finetuned-vit5/finetuned_ViT5/val_vit5.json")

print(f"[INFO] Total train records available: {len(df_train):,}")
if len(df_train) > MAX_TRAIN_SAMPLES:
    df_train = df_train.sample(n=MAX_TRAIN_SAMPLES, random_state=42).reset_index(drop=True)

if len(df_val) > MAX_VAL_SAMPLES:
    df_val = df_val.sample(n=MAX_VAL_SAMPLES, random_state=42).reset_index(drop=True)

train_dataset = Dataset.from_pandas(df_train)
val_dataset = Dataset.from_pandas(df_val)
print(f"[SUCCESS] Train dataset size: {len(train_dataset):,} | Val dataset size: {len(val_dataset):,}")

[INFO] Loading training data...
[INFO] Total train records available: 63,996
[SUCCESS] Train dataset size: 63,996 | Val dataset size: 3,369


In [5]:
# Khởi tạo Tokenizer và Tiền xử lý dữ liệu
print(f"[INFO] Initializing Tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def preprocess_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]
    
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("[INFO] Tokenizing datasets...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=["input_text", "target_text"], desc="Tokenizing Train Data")
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=["input_text", "target_text"], desc="Tokenizing Val Data")
print("[SUCCESS] Datasets tokenized successfully.")

[INFO] Initializing Tokenizer: VietAI/vit5-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[INFO] Tokenizing datasets...


Tokenizing Train Data:   0%|          | 0/63996 [00:00<?, ? examples/s]

Tokenizing Val Data:   0%|          | 0/3369 [00:00<?, ? examples/s]

[SUCCESS] Datasets tokenized successfully.


In [6]:
# Khởi tạo mô hình AutoModelForSeq2SeqLM và DataCollator
print(f"[INFO] Loading base model: {MODEL_ID}")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model.tie_weights()
print("[SUCCESS] Model loaded and tie_weights() called successfully.")

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

[INFO] Loading base model: VietAI/vit5-base


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

[SUCCESS] Model loaded and tie_weights() called successfully.


In [ ]:
# Khởi tạo các Callbacks (Theo dõi ETA & Đánh giá Generation giữa mỗi Epoch)
class DetailedProgressCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self.start_time = time.time()
        
    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n[INFO] ---> STARTING EPOCH {int(state.epoch) + 1} / {int(args.num_train_epochs)} <---")
        
    def on_epoch_end(self, args, state, control, **kwargs):
        elapsed_min = (time.time() - self.start_time) / 60.0
        epochs_done = int(state.epoch)
        total_epochs = int(args.num_train_epochs)
        est_total_min = (elapsed_min / epochs_done) * total_epochs if epochs_done > 0 else 0
        rem_min = max(0, est_total_min - elapsed_min)
        print(f"[INFO] <--- FINISHED EPOCH {epochs_done}/{total_epochs} | Elapsed: {elapsed_min:.1f} mins | Remaining ETA: ~{rem_min:.1f} mins --->")

class GenerationTestCallback(TrainerCallback):
    """Tests generation after each epoch to verify model can generate during training."""
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        was_training = model.training
        model.eval()
        test_q = "fix_asr: khi nào thì nên uống l cystin mỗi ngày có bị buồn lôn hay đau dạ dày không"
        inp = tokenizer(test_q, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_length=128, num_beams=1)
        decoded = tokenizer.decode(out[0], skip_special_tokens=True)
        raw = out[0].cpu().tolist()[:10]
        print(f"[GEN TEST] Epoch {int(state.epoch)} | Raw IDs: {raw} | Decoded: {decoded}")
        if was_training:
            model.train()

In [8]:
# Khởi tạo Seq2SeqTrainingArguments và Seq2SeqTrainer
# Lưu ý: Với transformers<4.40.0 (chuẩn NLP tham khảo), dùng evaluation_strategy="steps" và tokenizer=tokenizer
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False,                   # Khóa FP16 để ngăn overflow trên kiến trúc T5/ViT5
    max_grad_norm=1.0,            # Giới hạn gradient chống bùng nổ
    generation_max_length=128,
    logging_steps=200,
    report_to="none",
    load_best_model_at_end=False, # Tắt load_best_model_at_end để tránh lỗi untie weights khi reload checkpoint
    disable_tqdm=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[DetailedProgressCallback(), GenerationTestCallback()]
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [9]:
# Bắt đầu huấn luyện mô hình
print("[INFO] Starting Seq2Seq fine-tuning for ViT5 Medical Rewrite model...")
trainer.train()

[INFO] Starting Seq2Seq fine-tuning for ViT5 Medical Rewrite model...

[INFO] ---> STARTING EPOCH 1 / 3 <---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
1000,0.232600,0.177513
2000,0.175200,0.136768
3000,0.135600,0.119130
4000,0.125400,0.107129
5000,0.107400,0.104786
6000,0.102800,0.102242


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[INFO] <--- FINISHED EPOCH 1/3 | Elapsed: 41.1 mins | Remaining ETA: ~82.1 mins --->
[GEN TEST] Epoch 1 | Raw IDs: [0, 485, 3180, 35997, 4275, 35905, 35862, 1021, 847, 416] | Decoded: Fix_ASR: Khi nào thì nên uống L Cystin mỗi ngày có bị buồn nôn hay đau dạ dày không?

[INFO] ---> STARTING EPOCH 2 / 3 <---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[INFO] <--- FINISHED EPOCH 2/3 | Elapsed: 82.3 mins | Remaining ETA: ~41.1 mins --->
[GEN TEST] Epoch 2 | Raw IDs: [0, 485, 3180, 35997, 4275, 35905, 35862, 1021, 847, 416] | Decoded: Fix_ASR: Khi nào thì nên uống L Cystin mỗi ngày có bị buồn nôn hay đau dạ dày không?

[INFO] ---> STARTING EPOCH 3 / 3 <---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[INFO] <--- FINISHED EPOCH 3/3 | Elapsed: 123.3 mins | Remaining ETA: ~0.0 mins --->
[GEN TEST] Epoch 3 | Raw IDs: [0, 485, 3180, 35997, 4275, 35905, 35862, 1021, 847, 416] | Decoded: Fix_ASR: Khi nào thì nên uống L Cystin mỗi ngày có bị buồn nôn hay đau dạ dày không?


TrainOutput(global_step=6000, training_loss=0.17122622521718342, metrics={'train_runtime': 7397.2627, 'train_samples_per_second': 25.954, 'train_steps_per_second': 0.811, 'total_flos': 2.468710055098368e+16, 'train_loss': 0.17122622521718342, 'epoch': 3.0})

In [10]:
# Lưu mô hình và tokenizer sau huấn luyện
FINAL_MODEL_DIR = "/kaggle/working/vit5_medical_rewrite_final"
trainer.save_model(FINAL_MODEL_DIR)
model.tie_weights()
model.config.decoder_start_token_id = 0
model.config.eos_token_id = 1
model.config.pad_token_id = 0
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"[SUCCESS] Fine-tuning completed and best model saved to: {FINAL_MODEL_DIR}")

[SUCCESS] Fine-tuning completed and best model saved to: /kaggle/working/vit5_medical_rewrite_final


In [11]:
# Kiểm tra thực tế (Sanity Check) mô hình sau fine-tune với câu hỏi ASR mô phỏng
model.tie_weights()
model.eval()
print("\n" + "="*80)
print("[INFO] SANITY CHECK: TESTING FINE-TUNED VIT5 MODEL ON A SIMULATED NOISY QUESTION:")
test_noisy_q = "khi nào thì nên uống l cystin mỗi ngày có bị buồn lôn hay đau dạ dày không"
inputs = tokenizer(test_noisy_q, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
raw_ids = outputs[0].cpu().tolist()
pred_clean_q = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Input  (Noisy ASR)    : {test_noisy_q}")
print(f"Raw Token IDs (first 15): {raw_ids[:15]}")
print(f"Output (ViT5 Rewrite) : {pred_clean_q}")
print("="*80)

# Verify weight tying
print("[DEBUG] shared.weight is lm_head.weight:", model.shared.weight.data_ptr() == model.lm_head.weight.data_ptr())
print("[DEBUG] shared.weight norm:", model.shared.weight.data.norm().item())
print("[DEBUG] shared.weight[:3,:5]:", model.shared.weight.data[:3,:5].cpu().tolist())


[INFO] SANITY CHECK: TESTING FINE-TUNED VIT5 MODEL ON A SIMULATED NOISY QUESTION:
Input  (Noisy ASR)    : khi nào thì nên uống l cystin mỗi ngày có bị buồn lôn hay đau dạ dày không
Raw Token IDs (first 15): [0, 1021, 847, 416, 492, 1954, 75, 18043, 131, 1345, 198, 71, 138, 2897, 5820]
Output (ViT5 Rewrite) : Khi nào thì nên uống L Cystin mỗi ngày có bị buồn nôn hay đau dạ dày không?
[DEBUG] shared.weight is lm_head.weight: True
[DEBUG] shared.weight norm: 11584.4375
[DEBUG] shared.weight[:3,:5]: [[0.35392042994499207, 0.03056011162698269, 0.2551977038383484, 0.18664239346981049, -0.10793833434581757], [0.4279373586177826, 0.6897998452186584, 1.9320772886276245, 0.8760015964508057, -0.058498866856098175], [1.1637932062149048, 0.18873614072799683, -1.5625888109207153, 1.51570725440979, 0.7969162464141846]]
